# GISPR Module 8 — Tying It Together: End-to-End Workflows, GDAL, QGIS, GeoLibre & Google Earth Engine

**Course:** GIS Spatial Analysis with Python and R (GISPR)
**Module:** 8 — Integration, GDAL, and the Platform Landscape
**Builds on:** Modules 3–4 (vector), Modules 5–6 (raster), Module 7 (visualization)

---

### What you'll accomplish this module

By the end of this notebook you'll be able to:

| # | Learning outcome | Where |
|---|---|---|
| 1 | Integrate vector processing, raster analysis, and visualization into one repeatable, end-to-end spatial data science workflow | Sections 1, 6 |
| 2 | Execute GDAL operations at the command line — format conversion, reprojection, clipping — and explain how GDAL underpins `rasterio` (Python) and `terra` / `sf` (R) | Sections 2–4 |
| 3 | Compare QGIS, GeoLibre, Google Earth Engine, and programmatic Python/R workflows, and choose the right tool for a given task | Sections 5, 7 |

### Kernel reminder

- 🔵 **`[R]`** cells — switch kernel to **R** before running
- 🟢 **`[Python]`** cells — switch kernel to **Python 3** (or your cloned ArcGIS Pro env) before running
- 📋 **`[Terminal]`** cells — paste the command into your terminal / Anaconda Prompt, **not** in a notebook cell

> **The framing problem for this module:**
> A county wants to plant trees where they'll do the most good. Which neighborhoods are hottest in summer, have the least tree canopy, and are most socially vulnerable?
> Answering that needs raster data (land-surface temperature, canopy), vector data (neighborhoods, demographics), integration (zonal statistics + ranking), and a map to communicate it — the whole course, in one pipeline. Tonight you wire it together, add GDAL as the engine underneath it, and meet the professional power tools (QGIS, GeoLibre, Earth Engine) that sit alongside Python and R.

> **Nothing here is new spatial theory.** Module 8 is the wiring diagram for skills you already have, plus GDAL and the platform choices that make each stage faster. If you can name the seven stages below and which module taught each one, you're ready for tonight's lab.

---
## Section 1 — Anatomy of an End-to-End Workflow

Every real project — yours, ours, a consultant's — reduces to the same seven stages. Name them, and you can automate them.

| Stage | What happens | Module(s) | Typical tool |
|---|---|---|---|
| 1. Acquire | Download / API / DB | M2 | requests, ArcGIS API, Earth Engine |
| 2. Prepare | Format conversion, CRS, clean | M3, **M8 (GDAL)** | GDAL / OGR |
| 3. Vector ops | Buffer, clip, join | M4 | GeoPandas, sf |
| 4. Raster ops | Map algebra, zonal, warp | M5–6 | rasterio, terra |
| 5. Integrate | Combine, rank, model | M6–8 | pandas, dplyr |
| 6. Visualize | Static, interactive | M7 | matplotlib/tmap, folium/leaflet |
| 7. Communicate | Map, report, repo | M8+ | Quarto, GitHub |

**Reproducibility lives in the arrows.** Each hand-off between stages is code, not a click — so a script that runs stages 1→7 is one command to re-run when next quarter's data drops, instead of forty clicks you half-remember.

### Tonight's running scenario — Heat-Equity Screening

| Stage | This scenario |
|---|---|
| Acquire | Landsat summer land-surface temperature + NLCD canopy (raster); census block groups + a social-vulnerability index, or SVI (vector) |
| Prepare | Reproject everything to one projected CRS; clip rasters to the county boundary |
| Analyze | Zonal statistics — mean temperature and % canopy per block group |
| Integrate | Join demographics; rank block groups into a priority score |
| Communicate | A static map for the report, an interactive map for stakeholders |

The scenario is EJ-flavored on purpose — it mirrors the kind of screening EPA and local governments actually run (EJScreen, the Climate & Economic Justice Screening Tool). Every section from here builds one piece of this pipeline.

---
## Section 2 — GDAL: The Engine Under Everything

You've been using GDAL all along. When you called `rasterio.open()` in Module 5 or `rast()` in `terra`, GDAL did the actual reading. It's the C/C++ translation layer behind 200+ geospatial formats — GeoTIFF, IMG, NetCDF, GeoPackage, Shapefile, File Geodatabase, Cloud-Optimized GeoTIFF, and more.

| Library | Language | Layer |
|---|---|---|
| GDAL + PROJ + GEOS | C/C++ | The engine — format I/O, projections, geometry |
| `rasterio`, GeoPandas / Fiona | Python | Thin wrappers over GDAL |
| `terra`, `sf` | R | Thin wrappers over GDAL |
| QGIS, ArcGIS, PostGIS | Desktop / database | All call GDAL too |

- **PROJ handles projections.** Every reproject you've run — `st_transform()`, `.to_crs()`, `gdalwarp` — is PROJ, bundled with GDAL.
- **GEOS handles geometry.** Buffer, intersection, and the other geometry operations from Module 4 run through GEOS.

So why drop to the command line at all, when your Python/R functions already wrap GDAL? For fast, scriptable data prep with no notebook needed — convert a folder of rasters, reproject, and clip in a one-liner. That's next.

In [ ]:
# [Python] Confirm GDAL is the engine underneath rasterio
import rasterio

print("rasterio version:", rasterio.__version__)
print("GDAL version (via rasterio):", rasterio.__gdal_version__)

# Every rasterio driver name below is a GDAL driver name — same list gdalinfo --formats prints
with rasterio.Env() as env:
    drivers = rasterio.drivers.raster_driver_extensions()
print("\nA few formats rasterio/GDAL both read and write:")
for ext in ["tif", "img", "gpkg", "nc"]:
    print(f"  .{ext:<4} -> {drivers.get(ext, 'driver not registered in this build')}")


---
## Section 3 — GDAL at the Command Line

The same three operations — inspect, convert, reproject/clip — done the GDAL way and the library way. Learn the CLI once and it works everywhere: bash, Makefiles, CI, Docker.

We'll build the synthetic heat-equity rasters in Section 6, then run these exact commands against them. For now, read each command and match it to its rasterio/terra equivalent below.

### 3a — GDAL command-line reference

In [ ]:
# [Terminal] GDAL / OGR command-line reference — paste into a terminal, not a notebook cell

# 1 — Inspect: CRS, extent, bands, dtype, NoData
# gdalinfo lst_summer.tif

# 2 — Convert format (IMG -> compressed GeoTIFF)
# gdal_translate -of GTiff -co COMPRESS=DEFLATE canopy.img canopy.tif

# 3 — Reproject + resample to 30 m, UTM 10N
# gdalwarp -t_srs EPSG:32610 -tr 30 30 -r bilinear lst_summer.tif lst_utm.tif

# 4 — Clip a raster to a county polygon
# gdalwarp -cutline county.gpkg -crop_to_cutline lst_utm.tif lst_county.tif

# 5 — Rasterize a vector field to a grid
# gdal_rasterize -a svi -tr 30 30 blockgroups.gpkg svi.tif

# Vector conversion / reprojection uses ogr2ogr, GDAL's vector-side sibling
# ogr2ogr -t_srs EPSG:32610 bg_utm.gpkg blockgroups.shp

# GDAL 3.11+ adds a unified 'gdal' program: gdal raster info, gdal raster reproject,
# gdal vector convert. The classic tools above still work everywhere and fill most
# tutorials and Stack Overflow answers, so learn them first.

# Classic gotcha: -t_srs is the TARGET srs; -s_srs is the source. Mixing them up
# silently reprojects into the wrong CRS instead of erroring.


### 3b — The same three operations in Python (`rasterio`)

In [ ]:
# [Python] rasterio equivalents of gdalinfo / gdalwarp / gdal_translate
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.mask import mask as rio_mask

# gdalinfo -> just open it and read the profile
# with rasterio.open('lst_summer.tif') as src:
#     print(src.crs, src.bounds, src.res, src.dtypes)

# gdal_translate (format conversion) -> read profile, change driver/options, write
# with rasterio.open('canopy.img') as src:
#     profile = src.profile
#     profile.update(driver='GTiff', compress='DEFLATE')
#     with rasterio.open('canopy.tif', 'w', **profile) as dst:
#         dst.write(src.read())

# gdalwarp (reproject) -> rasterio.warp.calculate_default_transform + reproject
# with rasterio.open('lst_summer.tif') as src:
#     transform, width, height = calculate_default_transform(
#         src.crs, 'EPSG:32610', src.width, src.height, *src.bounds, resolution=30)
#     profile = src.profile.copy()
#     profile.update(crs='EPSG:32610', transform=transform, width=width, height=height)
#     with rasterio.open('lst_utm.tif', 'w', **profile) as dst:
#         reproject(source=rasterio.band(src, 1), destination=rasterio.band(dst, 1),
#                    src_transform=src.transform, src_crs=src.crs,
#                    dst_transform=transform, dst_crs='EPSG:32610',
#                    resampling=Resampling.bilinear)

# gdalwarp -cutline -crop_to_cutline (clip) -> rasterio.mask.mask
# import geopandas as gpd
# county = gpd.read_file('county.gpkg')
# with rasterio.open('lst_utm.tif') as src:
#     clipped, transform = rio_mask(src, county.geometry, crop=True)

print("See the commented cells above — each one is the library-side twin of a")
print("gdalinfo / gdal_translate / gdalwarp command from 3a. We'll run the real")
print("versions against synthetic data in Section 6.")


### 3c — The same three operations in R (`terra` / `sf`)

In [ ]:
# [R] terra / sf equivalents of gdalinfo / gdalwarp / gdal_rasterize
library(terra)
library(sf)

# gdalinfo -> just load it
# r <- rast('lst_summer.tif')
# r                              # prints CRS, extent, resolution, dtype

# gdalwarp reproject -> project()
# r2 <- project(r, 'EPSG:32610', method = 'bilinear')

# gdalwarp -cutline -crop_to_cutline -> crop() + mask()
# co <- vect('county.gpkg')
# clipped <- crop(r2, co, mask = TRUE)

# gdal_rasterize -> rasterize()
# bg <- vect('blockgroups.gpkg')
# svi_raster <- rasterize(bg, r2, field = 'svi')

# ogr2ogr reproject (vector) -> st_transform()
# bg_utm <- st_transform(st_read('blockgroups.shp', quiet = TRUE), 32610)

cat("terra and sf call GDAL under the hood for every one of these — project(),")
cat("crop(), rasterize(), and st_transform() are R's names for the same GDAL/PROJ\n")
cat("operations gdalwarp and gdal_rasterize run at the command line.\n")


### 3d — The professional pattern: calling GDAL from inside a Python/R pipeline

GDAL CLI and Python/R aren't rivals — they're layers. The move working analysts actually use is to shell out to the GDAL binary for the heavy, format-level lifting, then stay in code for the logic. That keeps a pipeline scriptable end to end without giving up GDAL's tuned, streaming C++ speed for big files.

In [ ]:
# [Python] Shelling out to GDAL from inside a pipeline with subprocess
import subprocess
import shutil

def gdal_available() -> bool:
    '''Return True if the gdalinfo binary is on PATH.'''
    return shutil.which("gdalinfo") is not None

def run_gdalwarp(src_path, dst_path, t_srs="EPSG:32610", tr=(30, 30), resampling="bilinear"):
    '''
    Wrap gdalwarp as a Python function — the pattern behind
    'a Python script that shells out to gdalwarp for the heavy prep.'
    '''
    cmd = [
        "gdalwarp", "-t_srs", t_srs,
        "-tr", str(tr[0]), str(tr[1]),
        "-r", resampling,
        src_path, dst_path,
    ]
    print("Running:", " ".join(cmd))
    return subprocess.run(cmd, check=True, capture_output=True, text=True)

if gdal_available():
    print("GDAL command-line tools found on PATH — run_gdalwarp() will work as written.")
else:
    print("GDAL command-line tools not found on PATH in this environment.")
    print("This is expected in many hosted Jupyter kernels — the function above is")
    print("still the pattern to use once GDAL is installed (conda install -c conda-forge gdal).")


### When to drop to the command line vs. stay in Python/R

| Reach for GDAL CLI when… | Stay in Python / R when… |
|---|---|
| Batch-processing a folder of files (bash loop or GNU parallel) | The step needs conditional logic, loops over analysis, or a model |
| One-off format conversion or reprojection — no notebook worth spinning up | You're combining vector + raster + tables in one reproducible flow |
| Files are huge and you want GDAL's tuned, streaming C++ speed | Results feed a plot, a report, or a dashboard in the same document |
| Wiring a step into a Makefile, shell script, CI job, or Dockerfile | You want unit-testable, version-controlled analysis others can re-run |
| You just need to inspect a file fast (`gdalinfo` / `ogrinfo`) | You need ArcPy / ArcGIS API integration alongside open-source steps |

In practice you mix them: a Python script that shells out to `gdalwarp` for the heavy prep, then does the analysis in `rasterio` — exactly what `run_gdalwarp()` above sets up. The pipeline is the boss; the CLI is one worker.

---
## Section 4 — The Platform Landscape

Four ways to do serious GIS work beyond your notebook. Each is a **category**, not just a product — know what each is best at, and where it hurts.

| Platform | Category | Best at | Watch out for |
|---|---|---|---|
| **QGIS** | Desktop GIS | Interactive exploration, cartography, quick QA of scripted output, no-code teammates. Free, open-source — the open ArcGIS Pro, with a Processing Toolbox, Model Builder, and a PyQGIS console. | Manual clicks aren't reproducible unless you script them. |
| **GeoLibre** | Cloud-native GIS | Lightweight cloud data, quick sharing, teams without local installs. Browser-based, reads COGs, GeoParquet, and STAC directly — no download, no install. Share a live map by URL. | Newer ecosystem; you're working against data in the cloud, not on disk. |
| **Google Earth Engine** | Cloud-scale analysis | Large-area / long time-series remote sensing without downloading data. Planetary-scale, petabyte imagery catalog (Landsat, Sentinel, MODIS, climate). Server-side compute; JavaScript editor + Python `ee` API. | Client vs. server execution model; harder to reproduce fully outside Google. |
| **Python / R** | Programmatic anchor | The pipeline that ties everything together, end to end. Full control, reproducible, version-controlled, automatable. Integrates ESRI (`arcpy`, ArcGIS API) and every open-source library — and can call all three tools above. | More upfront code than clicking; that's the price of reproducibility. |

### Choosing the right tool

There's no single best tool — only the best fit for the task in front of you.

| If you need to… | Reach for |
|---|---|
| Eyeball a file, make a quick map, hand it to a non-coder | QGIS |
| Batch-convert / reproject / clip a pile of files, fast | GDAL command line |
| Analyze decades of imagery over a whole region | Google Earth Engine |
| Explore cloud-native data (COG / STAC / GeoParquet) and share a link to a functional browser-based GIS | GeoLibre |
| Build a reproducible, automated, integrated pipeline — data in to decisions out | Python / R (the anchor) |

**The anchor mindset:** pick Python or R as home base, and call out to QGIS, GDAL, Earth Engine, or GeoLibre when they're the better worker for one stage. One pipeline, many tools.

---
## Section 5 — Guided Lab: Build the Heat-Equity Screening Pipeline

**Scenario:** A county wants to plant trees where they'll do the most good. Your job: rank every block group by summer heat, tree-canopy loss, and social vulnerability, then hand the planning office a map.

Work through steps 1–5 in order. Data note: to keep this notebook self-contained and download-free, we generate synthetic-but-realistic rasters and block-group polygons in step 0 — swap in your county's real Landsat LST, NLCD canopy, and block-group/SVI files and the rest of the pipeline runs unchanged.

**Goal:** a ranked priority map, plus a `run_pipeline()` function you could point at next summer's data.

### Step 0 — Build the synthetic study area (so this notebook runs standalone)

In [ ]:
# [Python] Step 0 — synthetic county, LST raster, canopy raster, and block groups with SVI
import os
import numpy as np
import rasterio
from rasterio.transform import from_origin
import geopandas as gpd
from shapely.geometry import box

os.makedirs("output", exist_ok=True)
np.random.seed(8)

# --- A 12 x 12 raster grid over a fictional county, 500 m cells, UTM 10N ---
ncols, nrows, cellsize = 12, 12, 500
xmin, ymax = 500000, 5275000     # arbitrary UTM 10N origin
transform = from_origin(xmin, ymax, cellsize, cellsize)
crs = "EPSG:32610"

# Summer land-surface temperature (°C): warmer in the urban core (top-left),
# cooler toward the river corridor (bottom-right) -- a believable heat-island pattern.
row_grad = np.linspace(34, 26, nrows).reshape(-1, 1)
col_grad = np.linspace(2, -2, ncols).reshape(1, -1)
lst = (row_grad + col_grad + np.random.normal(0, 0.6, (nrows, ncols))).astype("float32")

# Percent tree canopy (0-100): inverse-ish relationship with heat, plus noise
canopy = np.clip(70 - (lst - 26) * 6 + np.random.normal(0, 5, (nrows, ncols)), 0, 100).astype("float32")

def write_raster(path, array):
    profile = dict(
        driver="GTiff", height=nrows, width=ncols, count=1,
        dtype=array.dtype, crs=crs, transform=transform, nodata=-9999.0,
    )
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(array, 1)

write_raster("output/lst_summer.tif", lst)
write_raster("output/canopy.tif", canopy)
print("Wrote output/lst_summer.tif and output/canopy.tif")
print(f"  Grid: {nrows} x {ncols} cells @ {cellsize} m, CRS {crs}")

# --- County boundary: the full raster extent ---
county_geom = box(xmin, ymax - nrows * cellsize, xmin + ncols * cellsize, ymax)
county = gpd.GeoDataFrame({"name": ["Fictional County"]}, geometry=[county_geom], crs=crs)
county.to_file("output/county.gpkg", driver="GPKG", layer="county")

# --- Block groups: a 4x4 lattice of polygons, each with an SVI (social-vulnerability index) ---
bg_rows, bg_cols = 4, 4
bg_h, bg_w = (nrows * cellsize) / bg_rows, (ncols * cellsize) / bg_cols
records = []
for r in range(bg_rows):
    for c in range(bg_cols):
        x0 = xmin + c * bg_w
        y0 = ymax - (r + 1) * bg_h
        geom = box(x0, y0, x0 + bg_w, y0 + bg_h)
        # SVI trends higher toward the urban core (top-left), like many real cities
        svi = np.clip(0.85 - 0.12 * r - 0.10 * c + np.random.normal(0, 0.05), 0.05, 0.98)
        records.append({"bg_id": f"BG-{r}{c}", "svi": round(float(svi), 2), "geometry": geom})

blockgroups = gpd.GeoDataFrame(records, crs=crs)
blockgroups.to_file("output/blockgroups.gpkg", driver="GPKG", layer="blockgroups")
print(f"\nWrote output/blockgroups.gpkg -- {len(blockgroups)} block groups, SVI range "
      f"{blockgroups.svi.min():.2f}-{blockgroups.svi.max():.2f}")


### Step 1 — Prep with GDAL

In production this is where you'd reproject Landsat LST and NLCD canopy from their native CRS to a common UTM zone and clip both to the county boundary — the `gdalwarp` commands from Section 3a. Our synthetic rasters are already in UTM 10N and already county-sized, so we validate that instead of re-deriving it — the same `gdalinfo` sanity check you'd run first on real data.

In [ ]:
# [Python] Step 1 -- validate CRS + extent alignment before doing any analysis
# (the "assert it before analysis" fix for the #1 lab stall: CRS drift)

import rasterio
import geopandas as gpd

with rasterio.open("output/lst_summer.tif") as lst_src:
    lst_crs, lst_bounds, lst_res = lst_src.crs, lst_src.bounds, lst_src.res

with rasterio.open("output/canopy.tif") as canopy_src:
    canopy_crs = canopy_src.crs

blockgroups = gpd.read_file("output/blockgroups.gpkg")
county = gpd.read_file("output/county.gpkg")

print("LST      CRS:", lst_crs, "| resolution:", lst_res, "| bounds:", lst_bounds)
print("Canopy   CRS:", canopy_crs)
print("Blockgrp CRS:", blockgroups.crs)
print("County   CRS:", county.crs)

all_match = len({str(lst_crs), str(canopy_crs), str(blockgroups.crs), str(county.crs)}) == 1
print("\nAll four layers share one CRS:", "YES" if all_match else "NO -- reproject before continuing")

# Real-data equivalent you'd run at the terminal first:
#   gdalwarp -t_srs EPSG:32610 -tr 30 30 -r bilinear lst_raw.tif lst_utm.tif
#   gdalwarp -cutline county.gpkg -crop_to_cutline lst_utm.tif lst_summer.tif
#   ogr2ogr -t_srs EPSG:32610 blockgroups.gpkg blockgroups_raw.shp


### Step 2 — Raster analysis

Load the (clipped) rasters, mask NoData, and sanity-check with a quick map and histogram before going further — the same QC habit from Module 5.

In [ ]:
# [Python] Step 2 -- load rasters, mask NoData, sanity-check
import rasterio
import numpy as np
import matplotlib.pyplot as plt

with rasterio.open("output/lst_summer.tif") as src:
    lst = src.read(1, masked=True)
    lst_transform = src.transform

with rasterio.open("output/canopy.tif") as src:
    canopy = src.read(1, masked=True)

print("LST      -- min: {:.1f}C  max: {:.1f}C  mean: {:.1f}C".format(lst.min(), lst.max(), lst.mean()))
print("Canopy   -- min: {:.0f}%  max: {:.0f}%  mean: {:.0f}%".format(canopy.min(), canopy.max(), canopy.mean()))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

im0 = axes[0].imshow(lst, cmap="inferno")
axes[0].set_title("Summer LST (deg C)")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(canopy, cmap="Greens")
axes[1].set_title("Tree canopy (%)")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

axes[2].hist(lst.compressed(), bins=15, color="#4b2e83", alpha=0.85)
axes[2].set_title("LST distribution -- sanity check")
axes[2].set_xlabel("deg C")

plt.tight_layout()
plt.savefig("output/step2_raster_qc.png", dpi=150)
plt.show()
print("\nSaved output/step2_raster_qc.png")


### Step 3 — Zonal statistics

Summarize mean summer temperature and mean % canopy for each block-group polygon — the vector-meets-raster hand-off from Module 6.

In [ ]:
# [Python] Step 3 -- zonal statistics per block group (mean LST, mean canopy)
# pip install rasterstats --break-system-packages   # if not already installed
from rasterstats import zonal_stats
import geopandas as gpd

blockgroups = gpd.read_file("output/blockgroups.gpkg")

lst_stats = zonal_stats("output/blockgroups.gpkg", "output/lst_summer.tif", stats="mean", nodata=-9999.0)
canopy_stats = zonal_stats("output/blockgroups.gpkg", "output/canopy.tif", stats="mean", nodata=-9999.0)

blockgroups["mean_lst"] = [round(s["mean"], 2) if s["mean"] is not None else None for s in lst_stats]
blockgroups["mean_canopy"] = [round(s["mean"], 1) if s["mean"] is not None else None for s in canopy_stats]

print(blockgroups[["bg_id", "svi", "mean_lst", "mean_canopy"]].to_string(index=False))
blockgroups.to_file("output/blockgroups.gpkg", driver="GPKG", layer="blockgroups")


In [ ]:
# [R] Step 3 (R version) -- zonal statistics with terra::extract()
library(terra)
library(sf)

lst_r <- rast("output/lst_summer.tif")
canopy_r <- rast("output/canopy.tif")
bg <- vect("output/blockgroups.gpkg")

lst_means <- extract(lst_r, bg, fun = mean, na.rm = TRUE)
canopy_means <- extract(canopy_r, bg, fun = mean, na.rm = TRUE)

bg_df <- as.data.frame(bg)
bg_df$mean_lst <- round(lst_means[, 2], 2)
bg_df$mean_canopy <- round(canopy_means[, 2], 1)

print(bg_df[, c("bg_id", "svi", "mean_lst", "mean_canopy")])
# terra::extract() is the direct R equivalent of rasterstats.zonal_stats() --
# same GDAL/GEOS machinery underneath, different front door.


### Step 4 — Rank and join

Build a 0–1 priority score combining heat, low canopy, and high vulnerability, then sort to find the top-priority block groups for tree planting.

In [ ]:
# [Python] Step 4 -- normalize each factor to 0-1 and combine into a priority score
import geopandas as gpd

blockgroups = gpd.read_file("output/blockgroups.gpkg")

def normalize(series, invert=False):
    lo, hi = series.min(), series.max()
    norm = (series - lo) / (hi - lo)
    return 1 - norm if invert else norm

blockgroups["heat_score"] = normalize(blockgroups["mean_lst"])
blockgroups["canopy_score"] = normalize(blockgroups["mean_canopy"], invert=True)   # low canopy -> high score
blockgroups["svi_score"] = normalize(blockgroups["svi"])

# Equal-weighted composite -- document your weighting choice in the final report
blockgroups["priority_score"] = (
    blockgroups["heat_score"] + blockgroups["canopy_score"] + blockgroups["svi_score"]
) / 3

ranked = blockgroups.sort_values("priority_score", ascending=False)
print(ranked[["bg_id", "mean_lst", "mean_canopy", "svi", "priority_score"]]
      .round(2).to_string(index=False))

ranked.to_file("output/blockgroups.gpkg", driver="GPKG", layer="blockgroups")
print(f"\nTop priority block group: {ranked.iloc[0].bg_id}  (score {ranked.iloc[0].priority_score:.2f})")


### Step 5 — Map it

A static choropleth for the report, and an interactive map for stakeholders to explore.

In [ ]:
# [Python] Step 5a -- static priority map for the report
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

blockgroups = gpd.read_file("output/blockgroups.gpkg")

fig, ax = plt.subplots(figsize=(7, 7))
blockgroups.plot(
    column="priority_score", cmap="OrRd", legend=True,
    edgecolor="white", linewidth=1.2, ax=ax,
    legend_kwds={"label": "Tree-planting priority (0-1)", "shrink": 0.7},
)
# White text with a dark outline reads clearly across the whole OrRd ramp --
# a plain gray label (or plain white) loses contrast at one end or the other.
label_effects = [pe.withStroke(linewidth=2.5, foreground="#333333")]
for _, row in blockgroups.iterrows():
    c = row.geometry.centroid
    ax.annotate(
        row.bg_id, (c.x, c.y), ha="center", va="center",
        fontsize=8, color="white", path_effects=label_effects,
    )

ax.set_title("Heat-Equity Tree-Planting Priority by Block Group", fontsize=12, fontweight="bold")
ax.set_axis_off()
plt.tight_layout()
plt.savefig("output/priority_map_static.png", dpi=150)
plt.show()
print("Saved output/priority_map_static.png")


In [ ]:
# [Python] Step 5b -- interactive map for stakeholders (folium)
# pip install folium --break-system-packages   # if not already installed
import geopandas as gpd
import folium

blockgroups_utm = gpd.read_file("output/blockgroups.gpkg")           # projected CRS -- safe for centroid
center_pt = blockgroups_utm.union_all().centroid                     # compute center before reprojecting
blockgroups = blockgroups_utm.to_crs(epsg=4326)                      # folium needs WGS 84
center_pt_wgs84 = gpd.GeoSeries([center_pt], crs=blockgroups_utm.crs).to_crs(epsg=4326).iloc[0]

center = [center_pt_wgs84.y, center_pt_wgs84.x]
m = folium.Map(location=center, zoom_start=13, tiles="cartodbpositron")

folium.Choropleth(
    geo_data=blockgroups,
    data=blockgroups,
    columns=["bg_id", "priority_score"],
    key_on="feature.properties.bg_id",
    fill_color="OrRd",
    fill_opacity=0.75,
    line_opacity=0.6,
    legend_name="Tree-planting priority (0-1)",
).add_to(m)

folium.GeoJson(
    blockgroups,
    tooltip=folium.GeoJsonTooltip(fields=["bg_id", "mean_lst", "mean_canopy", "svi", "priority_score"]),
).add_to(m)

m.save("output/priority_map_interactive.html")
print("Saved output/priority_map_interactive.html -- open it in a browser to explore")


### Try it yourself

Wrap steps 1–5 into a single `run_pipeline(county, year)` function so next summer's data is one call away. That's the whole point of the module — reproducibility lives in the arrows, and a function is the arrow made concrete.

In [ ]:
# [Python] Try it yourself -- a reusable run_pipeline() wrapping steps 1-5
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats

def normalize(series, invert=False):
    lo, hi = series.min(), series.max()
    norm = (series - lo) / (hi - lo)
    return 1 - norm if invert else norm

def run_pipeline(county_name, lst_path, canopy_path, blockgroups_path, out_gpkg):
    '''
    Run the heat-equity screening pipeline end to end: zonal stats -> rank -> export.
    Assumes lst_path, canopy_path, and blockgroups_path are already reprojected and
    clipped (Step 1 -- do that with gdalwarp before calling this).
    '''
    print(f"[1/4] Loading block groups for {county_name}")
    bg = gpd.read_file(blockgroups_path)

    print("[2/4] Zonal statistics")
    bg["mean_lst"] = [s["mean"] for s in zonal_stats(bg, lst_path, stats="mean", nodata=-9999.0)]
    bg["mean_canopy"] = [s["mean"] for s in zonal_stats(bg, canopy_path, stats="mean", nodata=-9999.0)]

    print("[3/4] Scoring and ranking")
    bg["priority_score"] = (
        normalize(bg["mean_lst"]) + normalize(bg["mean_canopy"], invert=True) + normalize(bg["svi"])
    ) / 3
    bg = bg.sort_values("priority_score", ascending=False)

    print(f"[4/4] Writing {out_gpkg}")
    bg.to_file(out_gpkg, driver="GPKG", layer="priority")
    print(f"\nDone. Top priority block group: {bg.iloc[0].bg_id} (score {bg.iloc[0].priority_score:.2f})")
    return bg

# Re-run the whole pipeline in one call on the same synthetic data:
result = run_pipeline(
    county_name="Fictional County",
    lst_path="output/lst_summer.tif",
    canopy_path="output/canopy.tif",
    blockgroups_path="output/blockgroups.gpkg",
    out_gpkg="output/priority_automated.gpkg",
)


---
## Section 6 — Extended Application: Platforms & Your Project

Two halves: put a new platform through its paces on tonight's lab outputs, then start scoping the project you'll build for the rest of the course. Try **one** platform option below, then move to the proposal draft — everyone does the proposal eventually, since it's this module's graded deliverable.

### Option 1 — GDAL on your own data

Bring a raster or vector from your own work. Inspect it with `gdalinfo` / `ogrinfo`, then reproject and clip it with a single GDAL command. Time it against doing the same in ArcGIS Pro.

In [ ]:
# [Python] Option 1 -- run GDAL against a file of your own via subprocess
import subprocess
import shutil

my_file = "path/to/your_data.tif"   # <-- point this at your own raster or vector

if shutil.which("gdalinfo") is None:
    print("gdalinfo not found on PATH -- install with: conda install -c conda-forge gdal")
else:
    result = subprocess.run(["gdalinfo", my_file], capture_output=True, text=True)
    print(result.stdout if result.returncode == 0 else result.stderr)

# Once gdalinfo confirms the CRS, try the reproject + clip from Section 3a on your own file:
#   gdalwarp -t_srs EPSG:32610 -tr 30 30 -r bilinear your_data.tif your_data_utm.tif


### Option 2 — Earth Engine tour

Open the Earth Engine Code Editor (code.earthengine.google.com) or use the Python `ee` API below. Load a Landsat or Sentinel collection for your county, filter by date, and compute a median composite. Notice what never downloaded to your machine.

> **Environment note:** the `ee` package requires a free Earth Engine account and a one-time `ee.Authenticate()` browser sign-in. These cells are commented out for that reason — uncomment and run them once you're signed in.

In [ ]:
# [Python] Option 2 -- Google Earth Engine tour (requires a free EE account)
# pip install earthengine-api --break-system-packages   # if not already installed

# import ee
# ee.Authenticate()          # one-time browser sign-in
# ee.Initialize(project="your-gee-project-id")

# county_geom = ee.Geometry.Rectangle([-122.4, 47.4, -122.1, 47.7])   # swap in your county's bbox

# landsat = (
#     ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
#     .filterBounds(county_geom)
#     .filterDate("2024-06-01", "2024-08-31")
#     .filter(ee.Filter.lt("CLOUD_COVER", 20))
# )
# print("Scenes matching filter:", landsat.size().getInfo())

# composite = landsat.median().clip(county_geom)
# print("Median composite built server-side -- nothing downloaded yet.")

# Compare this to Section 6's Step 0-1: here the "acquire + prepare" stages run
# entirely on Google's servers against a petabyte catalog. That's the trade Earth
# Engine makes -- massive scale, but your reproducibility now depends on Google's
# platform rather than a file on disk.

print("Earth Engine cells are commented out -- uncomment after ee.Authenticate().")


### Option 3 — QGIS / GeoLibre check

Open your lab outputs in QGIS (or load the GeoPackage / a Cloud-Optimized GeoTIFF in GeoLibre). Style the priority map, add a legend, and export a layout — the cartography QGIS makes easy.

No code for this one — it's a hands-on check:

1. Open `output/priority_automated.gpkg` in QGIS or GeoLibre.
2. Style the `priority` layer by `priority_score` with a graduated color ramp.
3. Add a legend and a title, then export a layout (QGIS) or share the map's URL (GeoLibre).
4. Compare five minutes of point-and-click styling against the `matplotlib` map from Step 5 — which was faster? Which is more reproducible?

### Option 4 — Draft your final project proposal

Skip the tools and start the deliverable. Fill in each field below in your own words — this is your one-page proposal for the real-world workflow you'll build for the rest of the course, in R, Python, or both. It's the framework for your Module 12 final presentation.

#### Final Project Proposal — draft

*Replace each `_____` with your own answer. Keep the whole thing to about one page.*

**Problem**
> _____ (One real-world question worth answering with spatial analysis. Yours, ideally from work.)

**Data**
> _____ (The datasets you'll need and where they come from — note formats and CRSs.)

**Pipeline**
> _____ (The stages: acquire → prepare → vector → raster → integrate → visualize → communicate.)

**Tools**
> _____ (Python, R, or both — and any power tools, such as GDAL, GEE, QGIS, GeoLibre, or ArcPy, you'll call.)

**Deliverable**
> _____ (What you'll hand over: a map, a report, an interactive app, a reusable notebook.)

**Stretch goal**
> _____ (One thing you'll try if there's time — automation, a model, a web service.)

---

Submit as an R Markdown / Quarto / Jupyter notebook via Canvas and tag your repo `module8-proposal`. Due before Module 9. Questions → Ed Discussion #module8.

---

## 🔍 Module 8 — Self-Check Questions

Work through these without looking at the cells above. Then verify by running code or checking the deck.

**Workflow**

1. Name the seven stages of an end-to-end spatial workflow, in order. Which stage did Module 8 add tools for?
2. Why does the pipeline's reproducibility "live in the arrows" rather than in any single stage?
3. What's the difference between running your analysis once, correctly, and running it as a `run_pipeline()` function?

**GDAL**

4. What three libraries sit underneath `rasterio`, `terra`, and `sf`, and what does each one handle?
5. In `gdalwarp -t_srs EPSG:32610 -s_srs EPSG:4326 in.tif out.tif`, which flag is the target CRS and which is the source? What happens if you swap them?
6. Give two situations where you'd reach for the GDAL command line instead of writing a `rasterio` or `terra` script, and one where you'd do the opposite.
7. What does `subprocess.run(['gdalwarp', ...])` buy you that a pure-Python `rasterio.warp.reproject()` call doesn't?

**Platforms**

8. QGIS, GeoLibre, Google Earth Engine, and Python/R are described as four *categories*, not four products. What category does each belong to, and what's the one-line reason you'd pick each?
9. Why is Python/R called the "programmatic anchor" among the four platforms?
10. You need to analyze twenty years of Landsat imagery over three counties without downloading any of it. Which platform, and why?

**Integration**

11. Your zonal statistics step returns identical mean values for every block group — what's the most likely cause, and how would you check?
12. `gdalwarp` defaults to nearest-neighbor resampling; many Python/R functions default to bilinear. Why does this matter, and what's the fix?

---
## ✅ Module 8 Homework — Deliverables

Submit the following in Canvas before Module 9:

### Deliverable 1 — GDAL Command-Line Log (20 pts)
Pick **one** raster and **one** vector file of your own (or reuse the synthetic data from this notebook):
- Run `gdalinfo` / `ogrinfo` and record the CRS, extent, and (for the raster) band count and NoData value.
- Reproject both to a shared projected CRS with `gdalwarp` and `ogr2ogr`.
- Clip the raster to a boundary of your choice with `-cutline -crop_to_cutline`.

Submit: the exact commands you ran, in order, with one sentence per command explaining what it did and why you chose the flags you did.

### Deliverable 2 — The Heat-Equity Pipeline as a Reusable Function (30 pts)
Adapt `run_pipeline()` from Section 5 (or write your own) to a workflow that matters to you:
- The function must take at least three file paths as arguments (two data layers plus an output path).
- It must run zonal statistics joining a raster and a vector layer.
- It must compute at least one derived score or ranking column.
- It must export a GeoPackage.
- Commit the notebook and output to your GitHub repo; tag the commit `module8-pipeline`.

Submit: your GitHub repo link (same repo from prior modules).

### Deliverable 3 — Final Project Proposal (50 pts, graded)
Finish the proposal you drafted in Section 6, Option 4:
- One page, all six fields (Problem, Data, Pipeline, Tools, Deliverable, Stretch goal) filled in with specifics — a real question with real data beats an ambitious vague idea.
- Submit as an R Markdown / Quarto / Jupyter notebook via Canvas.
- Tag your repo `module8-proposal`.

This proposal becomes the framework for your Module 12 final presentation — scope will evolve as Modules 9–11 add modeling, live web services, and cloud-native formats, so it's fine to note open questions.

---

**Stuck?** Post in Ed Discussion — tag `#module8`. Include your error message, the cell that failed, and your OS/environment. If you solved something tricky, share how — the whole cohort benefits.